# **CNN**

In [9]:
import torch
from torchvision import transforms, models
import cv2
from PIL import Image
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load model
model = models.efficientnet_b0()
model.classifier[1] = nn.Linear(model.classifier[1].in_features, 15)

model.load_state_dict(torch.load("vehicle_color_best.pth", map_location=device))
model.to(device)
model.eval()

color_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

def predict_vehicle_color(crop):
    image = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB)
    image = Image.fromarray(image)
    image = color_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]

C:\Users\100ra\AppData\Local\Temp\ipykernel_24464\317031343.py:13: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("vehicle_color_best.pth", m

## Video inference

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
from sklearn.cluster import KMeans
from collections import defaultdict
from scipy.spatial import distance
import time

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

VEHICLE_CLASSES = {
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}

# LAB reference colors (perceptually stable)
COLOR_REFERENCES = {
    "red": (200, 30, 30),
    "blue": (30, 60, 200),
    "green": (40, 150, 40),
    "white": (230, 230, 230),
    "black": (30, 30, 30),
    "gray": (130, 130, 130),
    "silver": (180, 180, 180),
    "yellow": (230, 200, 30),
    "orange": (255, 140, 0),
    "brown": (120, 70, 40)
}

def map_rgb_to_color(rgb):
    min_dist = float("inf")
    best_color = "unknown"

    for color_name, ref in COLOR_REFERENCES.items():
        dist = np.linalg.norm(np.array(rgb) - np.array(ref))
        if dist < min_dist:
            min_dist = dist
            best_color = color_name

    return best_color

# --------------------------------------------------
# LOAD MODEL
# --------------------------------------------------

model = YOLO("yolov10m.pt")

# --------------------------------------------------
# COLOR DETECTION FUNCTION (ROBUST VERSION)
# --------------------------------------------------

def detect_vehicle_color(vehicle_crop):

    if vehicle_crop.size == 0:
        return "unknown"

    # Resize for speed
    small = cv2.resize(vehicle_crop, (100, 100))

    # Remove extreme dark/bright pixels
    hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
    mask = (hsv[:,:,2] > 40) & (hsv[:,:,2] < 240)
    pixels = small[mask]

    if len(pixels) < 100:
        return "unknown"

    # KMeans clustering
    kmeans = KMeans(n_clusters=7, n_init=10)
    kmeans.fit(pixels)

    centers = kmeans.cluster_centers_

    # Choose most dominant cluster
    counts = np.bincount(kmeans.labels_)
    dominant = centers[np.argmax(counts)]

    dominant = dominant.astype(int)  # BGR

    # Convert BGR → RGB
    dominant_rgb = dominant[::-1]


    return map_rgb_to_color(dominant_rgb)

# --------------------------------------------------
# VIDEO PIPELINE
# --------------------------------------------------

video_path = r"C:\Users\100ra\Downloads\Cars_Moving_On_Road_Stock_Footage_-_Free_Download_480p.mp4"
cap = cv2.VideoCapture(video_path)

vehicle_memory = defaultdict(dict)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True, verbose=False)

    if results[0].boxes.id is None:
        cv2.imshow("Vehicle System", frame)
        if cv2.waitKey(1) == 27:
            break
        continue

    boxes = results[0].boxes.xyxy.cpu().numpy()
    classes = results[0].boxes.cls.cpu().numpy()
    ids = results[0].boxes.id.cpu().numpy()

    for box, cls, track_id in zip(boxes, classes, ids):

        cls = int(cls)
        track_id = int(track_id)

        if cls not in VEHICLE_CLASSES:
            continue

        x1, y1, x2, y2 = map(int, box)
        vehicle_crop = frame[y1:y2, x1:x2]

        if vehicle_crop.size == 0:
            continue

        color = detect_vehicle_color(vehicle_crop)

        vehicle_memory[track_id]["type"] = VEHICLE_CLASSES[cls]
        vehicle_memory[track_id]["color"] = color
        vehicle_memory[track_id]["last_seen"] = time.time()

        label = f"ID:{track_id} {VEHICLE_CLASSES[cls]} {color}"

        print(label)
        print(f"Memory: {vehicle_memory[track_id]}")
        print(f'Color: {color}')



        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, label,
                    (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0,255,0), 2)

    cv2.imshow("Vehicle System", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

ID:1 car gray
Memory: {'type': 'car', 'color': 'gray', 'last_seen': 1771048518.8656611}
Color: gray
ID:2 car gray
Memory: {'type': 'car', 'color': 'gray', 'last_seen': 1771048518.922581}
Color: gray
ID:1 car gray
Memory: {'type': 'car', 'color': 'gray', 'last_seen': 1771048519.064767}
Color: gray
ID:2 car gray
Memory: {'type': 'car', 'color': 'gray', 'last_seen': 1771048519.1319273}
Color: gray
ID:1 car gray
Memory: {'type': 'car', 'color': 'gray', 'last_seen': 1771048519.248018}
Color: gray
ID:2 car gray
Memory: {'type': 'car', 'color': 'gray', 'last_seen': 1771048519.3060653}
Color: gray
ID:1 car gray
Memory: {'type': 'car', 'color': 'gray', 'last_seen': 1771048519.4176114}
Color: gray
ID:2 car gray
Memory: {'type': 'car', 'color': 'gray', 'last_seen': 1771048519.4747043}
Color: gray
ID:1 car gray
Memory: {'type': 'car', 'color': 'gray', 'last_seen': 1771048519.585953}
Color: gray
ID:2 car gray
Memory: {'type': 'car', 'color': 'gray', 'last_seen': 1771048519.641688}
Color: gray
ID:1 

## Image Inference

In [5]:
import os

# input_path = r"C:\Users\100ra\Downloads\4K Road traffic video for object detection and tracking - free download now!.mp4"
input_path = r"D:\Automatic ANPR\Self_Code\dataset_preprocessed\images\val\WB14.jpg"

# Check if file is image
image_extensions = [".jpg", ".jpeg", ".png", ".bmp"]

if os.path.splitext(input_path)[1].lower() in image_extensions:

    frame = cv2.imread(input_path)

    results = model.track(frame, persist=True, verbose=False)

    if results[0].boxes.id is not None:

        boxes = results[0].boxes.xyxy.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()
        ids = results[0].boxes.id.cpu().numpy()

        for box, cls, track_id in zip(boxes, classes, ids):

            cls = int(cls)
            if cls not in VEHICLE_CLASSES:
                continue

            x1, y1, x2, y2 = map(int, box)
            vehicle_crop = frame[y1:y2, x1:x2]

            color = detect_vehicle_color(vehicle_crop)

            label = f"ID:{int(track_id)} {VEHICLE_CLASSES[cls]} {color}"

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(frame, label, (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (0,255,0), 2)

    # Stay open until Q
    while True:
        cv2.imshow("Vehicle System", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cv2.destroyAllWindows()

else:
    # Treat as video
    cap = cv2.VideoCapture(input_path)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model.track(frame, persist=True, verbose=False)

        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            classes = results[0].boxes.cls.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy()

            for box, cls, track_id in zip(boxes, classes, ids):

                cls = int(cls)
                if cls not in VEHICLE_CLASSES:
                    continue

                x1, y1, x2, y2 = map(int, box)
                vehicle_crop = frame[y1:y2, x1:x2]

                color = detect_vehicle_color(vehicle_crop)

                label = f"ID:{int(track_id)} {VEHICLE_CLASSES[cls]} {color}"
                
                cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
                cv2.putText(frame, label, (x1, y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, (0,255,0), 2)
                print(label)
        cv2.imshow("Vehicle System", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()

## Web Based

In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
from sklearn.cluster import KMeans
from collections import defaultdict
from scipy.spatial import distance
import time

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

VEHICLE_CLASSES = {
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}

# LAB reference colors (perceptually stable)
REFERENCE_COLORS = {
    "red": [136, 208, 195],
    "dark_red": [120, 190, 180],
    "blue": [82, 207, 20],
    "sky_blue": [180, 140, 100],
    "green": [224, 42, 211],
    "yellow": [248, 106, 223],
    "orange": [191, 152, 207],
    "white": [255, 128, 128],
    "black": [0, 128, 128],
    "gray": [128, 128, 128],
    "silver": [192, 128, 128],
    "brown": [150, 160, 170],
    "maroon": [110, 170, 160]
}

# --------------------------------------------------
# LOAD MODEL
# --------------------------------------------------

model = YOLO("yolov10m.pt")

# --------------------------------------------------
# COLOR DETECTION FUNCTION (ROBUST VERSION)
# --------------------------------------------------

def detect_vehicle_color(vehicle_crop):

    h, w = vehicle_crop.shape[:2]

    # central region to avoid wheels / shadows
    crop = vehicle_crop[int(0.2*h):int(0.8*h),
                        int(0.2*w):int(0.8*w)]

    if crop.size == 0:
        return "unknown"

    blurred = cv2.GaussianBlur(crop, (5, 5), 0)

    # Convert to RGB
    rgb = cv2.cvtColor(blurred, cv2.COLOR_BGR2RGB)

    # Remove low saturation pixels
    hsv = cv2.cvtColor(blurred, cv2.COLOR_BGR2HSV)
    mask = hsv[:, :, 1] > 40
    pixels = rgb[mask]

    if len(pixels) < 50:
        return "unknown"

    # KMeans dominant color
    kmeans = KMeans(n_clusters=1, n_init=10)
    kmeans.fit(pixels)
    dominant_rgb = kmeans.cluster_centers_[0].astype("uint8")

    # Convert dominant color to LAB
    dominant_lab = cv2.cvtColor(
        np.uint8([[dominant_rgb]]),
        cv2.COLOR_RGB2LAB
    )[0][0]

    # Find nearest reference color
    min_dist = float("inf")
    best_color = "unknown"

    for color_name, ref_lab in REFERENCE_COLORS.items():
        d = distance.euclidean(dominant_lab, ref_lab)
        if d < min_dist:
            min_dist = d
            best_color = color_name

    return best_color

# --------------------------------------------------
# VIDEO PIPELINE
# --------------------------------------------------

cap = cv2.VideoCapture(0)

vehicle_memory = defaultdict(dict)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    results = model.track(frame, persist=True, verbose=False)

    if results[0].boxes.id is None:
        cv2.imshow("Vehicle System", frame)
        if cv2.waitKey(1) == 27:
            break
        continue

    boxes = results[0].boxes.xyxy.cpu().numpy()
    classes = results[0].boxes.cls.cpu().numpy()
    ids = results[0].boxes.id.cpu().numpy()

    for box, cls, track_id in zip(boxes, classes, ids):

        cls = int(cls)
        track_id = int(track_id)

        if cls not in VEHICLE_CLASSES:
            continue

        x1, y1, x2, y2 = map(int, box)
        vehicle_crop = frame[y1:y2, x1:x2]

        if vehicle_crop.size == 0:
            continue

        color = detect_vehicle_color(vehicle_crop)

        vehicle_memory[track_id]["type"] = VEHICLE_CLASSES[cls]
        vehicle_memory[track_id]["color"] = color
        vehicle_memory[track_id]["last_seen"] = time.time()

        label = f"ID:{track_id} {VEHICLE_CLASSES[cls]} {color}"

        cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
        cv2.putText(frame, label,
                    (x1, y1-10),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6, (0,255,0), 2)

    cv2.imshow("Vehicle System", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

## Prune Image Inference

In [7]:
from ultralytics import YOLO
import cv2
import numpy as np
from sklearn.cluster import KMeans
from collections import defaultdict
import os

# --------------------------------------------------
# CONFIG
# --------------------------------------------------

VEHICLE_CLASSES = {
    2: "car",
    3: "motorcycle",
    5: "bus",
    7: "truck"
}

LAB_REFERENCES = {
    "red":      (136, 208, 195),
    "blue":     (82, 207, 20),
    "green":    (224, 42, 211),
    "yellow":   (233, 80, 114),
    "white":    (255, 128, 128),
    "black":    (0, 128, 128),
    "gray":     (150, 128, 128),
    "silver":   (190, 128, 128),
    "orange":   (200, 150, 170),
    "brown":    (120, 150, 150)
}

# --------------------------------------------------
# LOAD MODEL
# --------------------------------------------------

model = YOLO("yolov10m.pt")

# --------------------------------------------------
# COLOR MAPPING
# --------------------------------------------------

def map_lab_to_color(lab_value):
    min_dist = float("inf")
    best = "unknown"

    for name, ref in LAB_REFERENCES.items():
        dist = np.linalg.norm(np.array(lab_value) - np.array(ref))
        if dist < min_dist:
            min_dist = dist
            best = name

    return best


def detect_vehicle_color(vehicle_crop):

    if vehicle_crop is None or vehicle_crop.size == 0:
        return "unknown"

    h, w = vehicle_crop.shape[:2]

    # Remove windshield + tires region
    crop = vehicle_crop[int(0.2*h):int(0.75*h),
                        int(0.15*w):int(0.85*w)]

    if crop.size == 0:
        return "unknown"

    crop = cv2.resize(crop, (120, 120))
    crop = cv2.GaussianBlur(crop, (5,5), 0)

    lab = cv2.cvtColor(crop, cv2.COLOR_BGR2LAB)
    pixels = lab.reshape((-1,3))

    # Remove extreme lightness pixels
    L_channel = pixels[:,0]
    mask = (L_channel > 40) & (L_channel < 220)
    pixels = pixels[mask]

    if len(pixels) < 200:
        return "unknown"

    kmeans = KMeans(n_clusters=3, n_init=10)
    kmeans.fit(pixels)

    centers = kmeans.cluster_centers_
    counts = np.bincount(kmeans.labels_)
    dominant = centers[np.argmax(counts)]

    return map_lab_to_color(dominant)

# --------------------------------------------------
# UNIVERSAL INPUT HANDLER
# --------------------------------------------------

input_path = r"C:\Users\100ra\Downloads\Cars_Moving_On_Road_Stock_Footage_-_Free_Download_480p.mp4"

image_extensions = [".jpg", ".jpeg", ".png", ".bmp"]

if os.path.splitext(input_path)[1].lower() in image_extensions:

    frame = cv2.imread(input_path)
    results = model(frame)

    if len(results) > 0:

        boxes = results[0].boxes.xyxy.cpu().numpy()
        classes = results[0].boxes.cls.cpu().numpy()

        for box, cls in zip(boxes, classes):

            cls = int(cls)
            if cls not in VEHICLE_CLASSES:
                continue

            x1, y1, x2, y2 = map(int, box)
            vehicle_crop = frame[y1:y2, x1:x2]

            color = detect_vehicle_color(vehicle_crop)

            label = f"{VEHICLE_CLASSES[cls]} {color}"

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
            cv2.putText(frame, label,
                        (x1, y1-10),
                        cv2.FONT_HERSHEY_SIMPLEX,
                        0.6, (0,255,0), 2)

    while True:
        cv2.imshow("Vehicle System", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cv2.destroyAllWindows()

else:

    cap = cv2.VideoCapture(input_path)

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model.track(frame, persist=True, verbose=False)

        if results[0].boxes.id is not None:

            boxes = results[0].boxes.xyxy.cpu().numpy()
            classes = results[0].boxes.cls.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy()

            for box, cls, track_id in zip(boxes, classes, ids):

                cls = int(cls)
                if cls not in VEHICLE_CLASSES:
                    continue

                x1, y1, x2, y2 = map(int, box)
                vehicle_crop = frame[y1:y2, x1:x2]

                color = detect_vehicle_color(vehicle_crop)

                label = f"ID:{int(track_id)} {VEHICLE_CLASSES[cls]} {color}"

                cv2.rectangle(frame, (x1, y1), (x2, y2), (0,255,0), 2)
                cv2.putText(frame, label,
                            (x1, y1-10),
                            cv2.FONT_HERSHEY_SIMPLEX,
                            0.6, (0,255,0), 2)

        cv2.imshow("Vehicle System", frame)

        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    cap.release()
    cv2.destroyAllWindows()